In [12]:
from typing import Literal
import numpy as np

rng = np.random.default_rng(42)

CellState = Literal["edge", "center"]
CellAction = Literal["death", "birth", "skip"]

class Cell:
    def __init__(self, cell_id: int, parent_id: int,
                 state: CellState,
                 x: int, y: int,
                 death_rate: float, birth_rate: float):
        self.cell_id = cell_id
        self.parent_id = parent_id
        self.state = state
        self.x, self.y = x, y
        self.death_rate = death_rate
        self.birth_rate = birth_rate

    def choose_action(self) -> CellAction:
        """Choose an action based on cells birth, death probability. returns "birth", "death", or "skip". returning "birth"
        does not mean the cell multiplies, it may be in a state (surrounded state) where it can't reproduce (under certain models)"""
        if rng.random() < self.death_rate:
            return "death"
        elif rng.random() < self.birth_rate:
            return "birth"
        return "skip"

class World:
    def __init__(self, size: int, time_step_hours: float = 2.0):
        self.cell_count = 0
        self.size = size
        self.grid = np.full((size, size), None, dtype=Cell)

        self.next_id = 1

        #to track sim time --> can print out for extra detail if needed/want
        self.step = 0
        self.time_step_hours = time_step_hours

    def update(self):
        # Every cell that exists at the start of update should be updated, in random order to not be biased
        cells = self.grid.flatten()
        cells = cells[cells != None]
        rng.shuffle(cells)

        for cell in cells:
            action = cell.choose_action()

            match action:
                case "death":
                    self.remove_cell(cell)
                case "birth":
                    self.birth_new_cell(cell)
            # print(f"cell {cell.cell_id} chooses to {action}")
        
        #update self.step and self.time_hours accordingly
        self.step += 1


    def _set_cell_at_position(self, cell: Cell):
        self.grid[cell.x,cell.y] = cell
        self.cell_count+=1

    def _get_next_cell_id(self):
        """return next unused id number and increment counter"""
        self.next_id += 1
        return self.next_id - 1

    def birth_new_cell(self, parent: Cell):
        """Look for a empty neighboring position from the parent cell and create a new Cell there"""
        empty_neighboring_indices = []
        for x in range(parent.x - 1, parent.x + 2):
            for y in range(parent.y - 1, parent.y + 2):
                if x == parent.x and y == parent.y:
                    continue
                if x < 0 or y < 0 or x >= self.size or y >= self.size:
                    continue
                if self.grid[x,y] is None:
                    empty_neighboring_indices.append((x,y))

        if not empty_neighboring_indices:
            # current cell is surrounded, for now, do nothing
            return

        cell_position = empty_neighboring_indices[rng.integers(len(empty_neighboring_indices))]
        self._set_cell_at_position(Cell(
            cell_id = self._get_next_cell_id(), parent_id = parent.cell_id,
            state ="edge",
            x =cell_position[0], y = cell_position[1],
            death_rate = parent.death_rate, birth_rate = parent.birth_rate
        ))


    def remove_cell(self, cell:Cell):
        self.grid[cell.x, cell.y] = None
        self.cell_count -= 1

# Helper Functions
def visualize_world(world: World):
    """hacky way to visualize each world grid"""
    grid = world.grid
    for row in grid:
        print("  ".join(["..." if x is None else f"{x.cell_id:03d}" for x in row]), flush=True)

In [15]:
from IPython.display import clear_output
from time import sleep

world = World(15)
world._set_cell_at_position(
    Cell(0, -1, "edge", 7, 7, .2, .8)
)

for i in range(15):
    world.update()
    clear_output(wait = True)

    print("Updates completed:", world.step)
    print("Simulated hours:", world.step * world.time_step_hours)
    print()

    visualize_world(world)
    print("\n")
    sleep(2)

print("*" * 10)
visualize_world(world)

Updates completed: 15
Simulated hours: 30.0

...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  ...  ...  ...  ...  181  ...  ...  ...  ...  ...
...  ...  ...  ...  ...  142  ...  ...  ...  113  ...  ...  ...  ...  ...
...  ...  ...  ...  165  ...  ...  168  ...  151  067  126  156  ...  ...
...  ...  ...  ...  161  073  060  136  087  182  094  120  162  ...  ...
...  ...  ...  187  130  115  039  052  184  013  071  107  ...  ...  ...
...  ...  ...  166  146  078  061  125  070  044  063  175  114  133  183
...  ...  ...  131  102  124  030  018  140  020  147  155  111  144  171
...  ...  ...  150  159  055  040  180  137  100  058  186  132  ...  ...
...  ...  ...  ...  ...  170  154  037  145  163  141  092  148  158  ...
...  ...  ...  ...  173  108  019  138  017  174  080  167  122  179  ...
...  ...  ...  ...  127  178  098  164  084  048  072  128  172  160  ...
...  ...  ...  ...  ...  076  059  057  109  177  ...  119  176  ..

In [16]:
for i in range(3):
    clear_output(wait=False)
    print(f"Frame {i}", flush=True)
    sleep(0.4)

Frame 2
